# Goal

Первый шаг на `frostbite_12`, в котором экшн план == коммит. 3 млн шагов, для паритета с докоммитной эрой. Учёт **только тренировочных** шагов (роллаутные не учитываем, т.к. там вслепую делается без обратной связи для агента) для паритета. Расширенный тестовый набор. 

По результатам `56.1d` ставим: экшн план некаузальный , `clip_coef=0.2`. По цифрам разницы нет, но зато так логичнее.

Сразу едем с `epochs_count=5` и `target_kl=0.1`.

Здесь разные `value_loss['clip_mode']`. Хочется закрыть этот вопрос (корректное вычисление `value`), т.к. он каскадно влиет на `advantage` -> `policy_loss`.

# Control plane

In [11]:
DOCKER_REGISTRY = 'develorium'

if DOCKER_REGISTRY is not None:
    import os
    os.environ['DOCKER_REGISTRY'] = DOCKER_REGISTRY
    
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_12.ipynb'
OPTUNA_DECISIVE_METRIC = 'test/levels_passed_mean'

# @launchit.collect
GRID_SEARCH_SPACE = dict(
    random_seed=list(range(1, 121, 10)),
    clip_mode=[
        None, 
        'legacy',
        'correct',
    ],
)
# @launchit.stop

if 'GRID_SEARCH_SPACE' in globals():
    import math
    LAUNCHES_COUNT = math.prod(map(len, GRID_SEARCH_SPACE.values()))
    assert LAUNCHES_COUNT > 0
else:
    LAUNCHES_COUNT = 12

# Manual override
# LAUNCHES_COUNT = 10 * 3 - 13

print(f'{LAUNCHES_COUNT=}')

LAUNCHES_COUNT=36


# set_hyperparameters

In [12]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = optuna_trial.suggest_categorical('random_seed', GRID_SEARCH_SPACE['random_seed'])
    HP.general.is_torch_deterministic = False
    HP.general.is_torch_compile = True
    
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6
    
    wm_parent = '18d_world_model_09:224'
    HP.vision_head.parent = dict(model=wm_parent, weights=wm_parent)
    HP.vision_head.is_trainable = False
    HP.vision_head.is_saving = False
    
    HP.encoder.parent = dict(model=wm_parent, weights=wm_parent)
    HP.encoder.is_trainable = True
    HP.encoder.is_saving = True

    HP.agent.parent = None 
    HP.agent.sequence_length = 33
    HP.agent.sequence_ages = [0, 1, 2, 3]
    HP.agent.action_plan_length = 10 
    HP.agent.is_causal_action_plan = False
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, dim_feedforward=1024, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.lora = None
    HP.agent.is_trainable = True
    HP.agent.is_saving = dict(weights=True, lora=False)
    
    # Test params
    HP.test.break_on_level_passed = True
    HP.test.envs_count = 30
    HP.test.env_stories = [
        ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+full_igloo+temperature_10+bailey_right_at_the_igloo_door'),
        ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+full_igloo+temperature_10+bailey_to_the_left_of_igloo'),
        ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+full_igloo+temperature_10'), # default initial Bailey's location
        ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+one_remaining_igloo+temperature_10+bailey_near_center'), 
        ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+three_remaining_igloo+temperature_20+bailey_near_center'), 
        ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+half_igloo+bailey_near_center'), 
        ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+half_igloo'), 
    ]
    HP.test.is_torch_amp = True
    
    # Rollout params
    HP.rollout.steps_count = 64 # 64 * 10 (action_plan_length) -> 640 actual steps for each env per rollout
    HP.rollout.env_stories = {
        '0:4':   ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+full_igloo+temperature_10+bailey_right_at_the_igloo_door'), 
        '4:8':   ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+full_igloo+temperature_10+bailey_very_near_igloo_door'), 
        '8:12':  ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+full_igloo+temperature_10+bailey_near_center'), 
        '12:16': ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+one_remaining_igloo+temperature_10+bailey_near_center'), 
        '16:20': ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+three_remaining_igloo+temperature_20+bailey_near_center'), 
        '20:24': ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+half_igloo+bailey_near_center'), 
        '24:28': ('com.develorium.neurolab.frostbite_ram:level1:1:cls=none', 'no_score+last_life+half_igloo'), 
    }
    HP.rollout.is_per_step_advantage = True
    HP.rollout.is_torch_amp = True
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000
    
    HP.ppo.epochs_count = 5 
    HP.ppo.batch_size = 448
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.policy_loss = dict(is_per_slot_clipping=False)
    HP.ppo.value_loss = dict(coef=0.2, clip_mode=optuna_trial.suggest_categorical('clip_mode', GRID_SEARCH_SPACE['clip_mode']), clip_delta=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.05)')
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.2
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = 0.1 # e target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    
    HP.ppo.is_torch_amp = True
    
    # Reporting
    HP.report.rollout_env_groups = None
    HP.report.follow_up_freq = 10

    return HP

# Results

Самый перспективный режим `clip_mode=None`. Никаких обрезок, критик получает градиенты, часто и по делу. Быстрее всех выучиывается и точнее, каскадно распроняя своё влияние. Режим `correct` самый хреновый - режет много. Но это может быть из-за заниженного `clip_delta=0.2`. По идее `clip_delta` должен быть согласован с масштабом обрезаемого значения, но подбором этого параметра заморачиваться не хочется. `clip_mode=legacy` посередине. Если смотреть со стороны, то этот режим рандомно гасит градиенты, но делает это много реже, чем `correct`.

| метрика | 3 режима `clip_mode` |
|---|---|
| `rollout/episode_r` | <img src="./img/episode_r_56.1i.png"> |
| `rollout/episode_lp` | <img src="./img/episode_lp_56.1i.png"> |
| `test/levels_passed_mean` | <img src="./img/levels_passed_mean_56.1i.png"> |
| `losses/c) value_loss` | <img src="./img/value_loss_56.1i.png"> |

**Решающая метрика в домашнем формате** (`test/levels_passed_mean`, 12 ранов на режим):

| | `None` (чистый `MSE`) | `legacy` (`max`) | `correct` (`where`-блок) |
|---|---|---|---|
| Sample Size (n) | 12 | 12 | 12 |
| Mean Score | **0.5905** | 0.5440 | 0.5373 |
| Best Value | 0.7952 (`18n_ppo_tr_frostbite_12:321`) | 0.6905 (`:337`) | 0.6571 (`:335`) |
| Standard Deviation | 0.1257 | 0.0680 | 0.0837 |
| Shapiro-Wilk (p) | 0.410 | 0.089 | 0.022 |
| Welch's t-test (p) | — (референс) | 0.276 | 0.237 |
| Mann-Whitney U (p) | — | 0.707 | 0.707 |
| Cohen's d | — | −0.459 | −0.498 |
| Вердикт | референс | н/з | н/з |

**Выводы**
1) впредь надо использовать None и не париться.

**Подробности**

| метрика | `None` | `legacy` | `correct` | `legacy`−`None` | `correct`−`None` | `correct`−`legacy` |
|---|---|---|---|---|---|---|
| `test/levels_passed_mean` (last) | 0.590 ± 0.126 | 0.544 ± 0.068 | 0.537 ± 0.084 | Δ−0.046, p=0.37 | Δ−0.053, p=0.25 | Δ−0.007, p=0.77 |
| `test/levels_passed_mean` (max) | 0.648 ± 0.119 | 0.564 ± 0.064 | 0.578 ± 0.037 | Δ−0.084, p=0.074 | Δ−0.070, **p=0.049** | Δ+0.014, p=0.52 |
| `rollout/episode_r` (last) | 788.5 ± 346.2 | 748.5 ± 144.3 | 525.5 ± 127.2 | Δ−40, p=0.73 | Δ−263, **p=0.013** (1/11) | Δ−223, **p=0.0002** (0/12) |
| `rollout/episode_lp` (last) | 0.440 ± 0.171 | 0.374 ± 0.066 | 0.320 ± 0.065 | Δ−0.066, p=0.21 | Δ−0.119, **p=0.027** (2/10) | Δ−0.053, **p=0.042** |
| `losses/i) epochs` (mean) | 3.31 ± 0.19 | 3.21 ± 0.43 | 3.34 ± 0.17 | Δ−0.10, p=0.47 | Δ+0.03, p=0.78 | Δ+0.13, p=0.31 |
| `losses/c) value_loss` (mean) | 2.90 ± 1.17 | 3.57 ± 0.80 | 4.69 ± 0.77 | Δ+0.67, p=0.19 | Δ+1.79, **p=0.0013** (11/1) | Δ+1.12, **p=0.013** (9/3) |

| **механизм (TB)** | `None` | `legacy` | `correct` |
|---|---|---|---|
| `grad/a) total_by_value_loss` (mean) | 2.662 ± 0.378 | 1.525 ± 0.214 | 0.299 ± 0.034 |
| `rollout/c) value_mean` (mean) | 16.97 | 16.17 | 13.15 |
| `rollout/d) advantage_mean` (mean) | +0.018 | +0.011 | −0.013 |
| `test/returns_mean` (mean) | 161.1 | 153.4 | 141.6 |

**По решающей метрике (`test/levels_passed_mean`, last) разницы между режимами обрезки `value_loss` НЕТ** (0.590 / 0.544 / 0.537; парные p = 0.37 и 0.25). Но по непрерывным каналам (`rollout/episode_r`, `rollout/episode_lp`) обрезка вредит, и «правильная» реализация (`correct` — жёсткий блок строк вне коридора) вредит **сильнее** «сломанной» (`legacy` — `max` из baselines): `correct` против `legacy` по `episode_r` даёт Δ = −223 при парном p = 0.0002 и 0/12 побед. Механизм — **градиентное голодание критика**: средний `grad/a) total_by_value_loss` = 2.66 (`None`) → 1.53 (`legacy`) → 0.30 (`correct`), все пары p = 0.0005. Фактическое число эпох обучения режим не двигает (3.31 / 3.21 / 3.34, н/з).

**Состояние систематической проблемы** (корректное вычисление `value`, каскад `value → advantage → policy`): вопрос `clip_mode` **закрыт измерениями** — на решающей метрике разницы нет; на непрерывных каналах обрезка (любая) не помогает, а её «правильная» форма (`correct`) — худшая. Оговорка: per-row эффект на критика по метрикам не инструментирован (метрики вида `value clip frac` в логах нет), эффект оценён косвенно через `grad/a) total_by_value_loss`.

**Подробные выводы**
1) **На решающей метрике режимы неразличимы** (0.590 / 0.544 / 0.537, н/з). Разница, если и есть, — ниже разрешения n=12 на этом шаге.
2) **На непрерывных каналах обрезка вредит, и `correct` — худший**: `correct` глушит ~90 % градиента критика (0.30 против 2.66 у `None`), критик не доучивается → `value_mean`/`advantage` хуже → `episode_r`/`episode_lp` просаживаются. `legacy` (наследие baselines, `max`) мягче, но всё равно не лучше `None`.
3) **`epochs` режим не двигает** — обрезка не трогает поведение early-stop/`target_kl`. Рост `value_loss` при обрезке — цензурированный фит (критик не может войти в коридор), сам по себе не сигнал.
4) **Для дальнейших шагов — `clip_mode=None`** (чистый `MSE`): не хуже на решающей метрике и лучше/равно на всех непрерывных каналах, плюс полный градиент критика. Совпадает с литературой (Engstrom 2020; Andrychowicz 2021, decision C13: value clipping вредит).

# System

In [13]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [14]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_12.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_12',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_56.1i/18n_study_56.1i.ipynb',
 'optuna_study_name': '18n_study_56.1i',
 'optuna_study_serial': '56.1i',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_56.1i/18n_study_56.1i.optuna'}

In [15]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [16]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [17]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [18]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [19]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while LAUNCHES_COUNT is None or completed_launches_count < LAUNCHES_COUNT:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if LAUNCHES_COUNT is None or (completed_launches_count + len(futures) < LAUNCHES_COUNT):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-11-12 16:39:31,566] A new study created in Journal with name: 18n_study_56.1i


2026.11.12-16:39:32.075834     0.485 >> Model instance registered, version=314
2026.11.12-16:39:32.089314     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_12-launch314.ipynb"
2026.11.12-16:39:32.089822     0.006 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_12:314"; running launches=1
2026.11.12-16:40:03.327310     0.535 >> Model instance registered, version=315
2026.11.12-16:40:03.346704     0.007 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_12-launch315.ipynb"
2026.11.12-16:40:03.347156     0.007 >> 5.5 idle runners exist, submitted launch "18n_ppo_tr_frostbite_12:315"; running launches=2
2026.11.12-16:40:34.477379     0.420 >> Model instance registered, version=316
2026.11.12-16:40:34.488642     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_12-launch316.ipynb"
2026.11.12-16:40:34.489007     0.006 >> 4.8 idle runners exist, submitted launch "18n_ppo_tr_frostbite_12

In [20]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-11-13 00:44:46,098] Using an existing study with name '18n_study_56.1i' instead of creating a new one.


2026.11.13-00:44:46.101041     5.038 >> Study statistics: 
2026.11.13-00:44:46.106567     0.006 >> 	Number of finished trials: 36
2026.11.13-00:44:46.107360     0.001 >> 	Number of pruned trials: 0
2026.11.13-00:44:46.108011     0.001 >> 	Number of complete trials: 36
2026.11.13-00:44:46.108930     0.001 >> Best trial:
2026.11.13-00:44:46.110301     0.001 >> 	Value: 0.7952380952380952
2026.11.13-00:44:46.110889     0.001 >> 	Model version: 321
2026.11.13-00:44:46.111344     0.000 >> 	Params: 
2026.11.13-00:44:46.111885     0.001 >> 		random_seed: 21
2026.11.13-00:44:46.112257     0.000 >> 		clip_mode: None


In [21]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.11.13-00:44:46.118514     0.006 >> Top trials:
2026.11.13-00:44:46.119076     0.001 >> 1) 18n_ppo_tr_frostbite_12:321, value=0.7952380952380952
2026.11.13-00:44:46.119376     0.000 >> 2) 18n_ppo_tr_frostbite_12:329, value=0.7904761904761904
2026.11.13-00:44:46.119553     0.000 >> 3) 18n_ppo_tr_frostbite_12:337, value=0.6904761904761905


In [22]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(0.5572751322751323), np.float64(0.09454433900121682))